In [ ]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [6]:
import asyncio
import json
import warnings
warnings.filterwarnings('ignore')
from tqdm.asyncio import tqdm
from dotenv import load_dotenv
from evaluation.evaluator import evaluation_agent, format_agent_turns_to_markdown

load_dotenv()

/Users/adityachhabra/Github/openagrinet/oan-evaluation


True

Logfire project URL: 
]8;id=381496;https://logfire-us.pydantic.dev/kenpath-aditya/sunbird-va-dev\https://logfire-us.pydantic.dev/kenpath-aditya/sunbird-va-dev]8;;\


In [11]:
model_name  = "openai_gpt-oss-20b"
MODEL_DATA_PATH = os.path.join("data", "models", model_name.replace("/", "_"))
json_file = os.path.join(MODEL_DATA_PATH, "data.json")
eval_json_file = os.path.join(MODEL_DATA_PATH, "evaluation.json")   

In [12]:
# Load data
with open(json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)


In [15]:
import random

In [17]:
item = random.choice(data)

In [22]:
async def evaluate_item(item):
    """Evaluate a single item with concurrency control."""
    question = item.get("question", "")
    
    # Format message
    agent_turns = format_agent_turns_to_markdown(item.get("agent_turns", []))
    message = "\n\n".join([
        "*User Question:*\n" + question,
        "*Category:*\n" + item.get("category", ""),
        "*Agristack Required:*\n" + item.get("agristack_required", "No"),
        "*Agent Turns:*\n" + agent_turns,
        "*Final Response:*\n" + item.get("answer", "")
    ])
    
    # Evaluate
    eval_result = await evaluation_agent.run(message)
    item['evaluation'] = eval_result.output.model_dump()
    return item
    

In [23]:
resp = await evaluate_item(item)

02:18:03.215 Evaluation Agent run
02:18:03.224   chat gpt-5


In [25]:
resp['evaluation']

{'protocol': {'scope_discipline': {'score': 5,
   'explanation': 'The agent stayed within agricultural scope, answering a soil testing fee question.',
   'evidence': ['User asks about soil micronutrient testing government fee; answer remains on soil testing/fees.']},
  'mandatory_tool_compliance': {'score': 0,
   'explanation': 'Agristack Required was Yes but no Agristack/profile fetch was attempted.',
   'evidence': ['Meta: Agristack Required: Yes',
    'Trace shows only search_documents calls; no Agristack/profile tool usage.']},
  'agristack_workflow_compliance': {'score': 0,
   'explanation': 'No use of farmer profile/location or holdings to tailor service/fees; no subsequent localized tool calls.',
   'evidence': ['No profile fetch, no geolocation/service lookup in trace.']},
  'term_identification_compliance': {'score': 1,
   'explanation': 'Did not use search_terms before searching; jumped straight to document search.',
   'evidence': ['Trace shows search_documents called multip

In [ ]:

# Load existing evaluations
existing = {}
if os.path.exists(eval_json_file):
    with open(eval_json_file, 'r', encoding='utf-8') as f:
        for item in json.load(f):
            q = item.get("question", "")
            if q and "evaluation" in item:
                existing[q] = item
    print(f"Loaded {len(existing)} existing evaluations")

# Semaphore to limit concurrent requests
MAX_CONCURRENT = 10
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

async def evaluate_item(item):
    """Evaluate a single item with concurrency control."""
    async with semaphore:
        question = item.get("question", "")
        
        # Format message
        agent_turns = format_agent_turns_to_markdown(item.get("agent_turns", []))
        message = "\n\n".join([
            "*User Question:*\n" + question,
            "*Category:*\n" + item.get("category", ""),
            "*Agristack Required:*\n" + item.get("agristack_required", "No"),
            "*Agent Turns:*\n" + agent_turns,
            "*Final Response:*\n" + item.get("answer", "")
        ])
        
        # Evaluate
        try:
            eval_result = await evaluation_agent.run(message)
            item['evaluation'] = eval_result.output.model_dump()
            return item
        except Exception as e:
            print(f"Error: {question[:50]}... - {e}")
            return None

async def main():
    # Separate items into already evaluated and to-be-evaluated
    results = []
    tasks = []
    
    for item in data:
        question = item.get("question", "")
        
        # Skip if already evaluated
        if question in existing:
            results.append(existing[question])
        else:
            tasks.append(evaluate_item(item))
    
    print(f"Skipping {len(results)} already evaluated items")
    print(f"Evaluating {len(tasks)} items in parallel (max {MAX_CONCURRENT} concurrent)...")
    
    # Run all evaluations in parallel with progress bar
    if tasks:
        completed = await tqdm.gather(*tasks, desc="Evaluating")
        
        # Add successful evaluations to results
        for result in completed:
            if result is not None:
                results.append(result)
    
    # Save all results
    with open(eval_json_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
    
    return results

if __name__ == "__main__":
    asyncio.run(main())
    print("Evaluation Done!")

In [16]:
import os
from pathlib import Path
from pydantic import BaseModel, Field
from pydantic_ai import Agent
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [17]:
Path = r"D:\Kenpath\oan-evaluation"

In [18]:
os.chdir(Path)

In [19]:
from dotenv import load_dotenv
load_dotenv()


True

In [20]:
from helpers.utils import get_prompt

In [21]:
class Metric(BaseModel):
    farmer_id : str = Field(..., description="The id of the farmer")
    question: str = Field(..., description="The question that the user asked")
    response: str = Field(..., description="The response from the model")
    metric_name: str = Field(..., description="The name of the metric to evaluate")
    score: int = Field(..., description="The score of the metric", ge=0, le=5)
    explanation: str = Field(..., description="The explanation of the score")

In [22]:
class EvaluationResult(BaseModel):
    """Result of the evaluation."""
    source_attribution: Metric = Field(..., description="Measures how well the response references appropriate, authoritative, and relevant sources. Higher scores reflect correct alignment between cited sources and the information presented.")
    grammatical_accuracy: Metric = Field(..., description="Evaluates the degree to which the response maintains grammatical correctness, valid word usage, accurate translations or transliterations, and proper sentence structure in Marathi.")
    factual_correctness: Metric = Field(..., description="Measures the accuracy of all factual information in the response, including details related to government schemes, mandi prices, fertilizers, pesticides, quantities, timelines, and agricultural guidelines, without introducing incorrect or misleading claims")
    actionability: Metric = Field(..., description="Evaluates the extent to which the response provides practical and implementable guidance, such as quantities, ratios, step-by-step instructions, timelines, or decision criteria that can be acted upon directly by the user.")
    completeness: Metric = Field(..., description="Measures how thoroughly the response addresses the user's query, including coverage of essential steps, conditions, data points, constraints, and safety considerations where relevant.")
    context_alignment: Metric = Field(..., description="Measures how closely the response stays aligned with the user's question and situational context, avoiding unnecessary or unrelated information, and introducing follow-up questions only when contextually justified.")
    information: Metric = Field(..., description="Evaluates whether the level of detail is suitable for the intended user, balancing clarity and precision without being overly technical or overly vague, and reflecting real-world agricultural practices.")
    safety_compliance: Metric = Field(..., description="Measures adherence to regulatory and safety norms by avoiding recommendations of banned, unsafe, or non-compliant agricultural inputs or practices, and by including appropriate safety precautions where applicable.")
    tool_usage: Metric = Field(..., description="Measures whether the response reflects correct selection and sequencing of tools or processes as required by the task, without omission, misuse, or unnecessary invocation.")
    retrieval_grounding: Metric = Field(..., description="Evaluates the extent to which the response is grounded in retrieved or provided documents and avoids introducing unsupported, speculative, or fabricated information.")

In [23]:
evaluation_agent = Agent(
    model='gpt-5',
    name="Evaluation Agent",
    instrument=True,
    output_type=EvaluationResult,
    retries=2,
    system_prompt=get_prompt('evaluation_system'),
)

For logfire span

In [ ]:
text = """User Query: सौर कुंपण योजना
Language: Marathi
==================================================

Assistant called tool: get_scheme_codes
Arguments: {}

Tool 'get_scheme_codes' returned:
## State Schemes (Maharashtra)

| Scheme Name | Scheme Code |
|-------------|-------------|
| Nanaji Deshmukh Krishi Sanjivani Prakalp Individual Farm Ponds | ndksp-individual-farm-ponds |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Drip Irrigation | ndksp-drip-irrigation |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Vegetable Planting Material in Shednet and Polyhouse | ndksp-vegetable-planting-material-in-shednet-and-polyhouse |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sericulture | ndksp-sericulture-unit |
| Chief Minister Sustainable Agriculture Irrigation Individual Farm Pond | cmsaisfp |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Seed Production | ndksp-seed-production |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Inland Fishery | ndksp-inland-fishery |
| Dr. Babasaheb Ambedkar Krushi Swavalamban Yojna | baksy |
| Gopinath Munde Shetkari apghat suraksha sanugrah anudan yojna | gmsassay |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Sprinkler Irrigation | ndksp-sprinkler-irrigation |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Pipes | ndksp-pipes |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Organic Production Unit | ndksp-organic-production-unit |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Pump Set | ndksp-pump-set |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Goat Rearing | ndksp-goat-rearing |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Well Recharge | ndksp-well-recharge |
| Bhausaheb Fundkar Horticulture Plantation Scheme | bfhps |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Planting Material Polyhouse | ndksp-flower-crop-planting-material-polyhouse |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Horticulture | ndksp-horticulture-plantation |
| Birsa Munda Krushi Kranti Yojna (TSP and OTSP) | bmkky |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Farm Pond Lining | ndksp-farm-pond-lining |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Agroforestry | ndksp-agroforestry-tree |
| Nanaji Deshmukh Krishi Sanjivani Prakalp Bamboo | ndksp-agroforestry-bamboo |
| Namo Shetkari Mahasanman Nidhi (NSMNY) | nsmnyy |

## Central Schemes

| Scheme Name | Scheme Code |
|-------------|-------------|
| Mahatma Gandhi National Rural Employment Guarantee Scheme | mgnregs |
| Pradhan Mantri Kisan Mandhan Yojana | pmkmy |
| Krishonnati Yojana-Mission for Integrated Development of Horticulture | kymidh |
| PM-RKVY-Sub Mission on Agriculture Mechanization | pmrkvysmam |
| Pradhanmantri Kisan Samman Nidhi Yojna | pmkisan |
| Restructured Weather Base Crop Insurance | rwbcis |
| Pradhan Mantri Fasal Bima Yojna (PMFBY) | pmfby |
| Prime Minister Formalisation of Micro Food Processing Enterprises | pmfmfpes |
| Agriculture Infrastructure Fund (AIF) | aif |
| PM-RKVY-Per Drop More Crop | pmkrvypdmc |
...
==================================================
==================================================

Final Response:
आपण विचारलेली "सौर कुंपण योजना" (Solar Fencing Scheme) नावाची कृषी योजना सद्यस्थितीत महाराष्ट्रातील अधिकृत शासकीय योजना यादीमध्ये स्वतंत्रपणे उपलब्ध नाही. सध्या शेतकऱ्यांसाठी देण्यात येणाऱ्या महत्त्वाच्या राज्य व केंद्र सरकारच्या योजना वरील यादीत दिल्या आहेत. त्यामध्ये सिंचन, शेती यंत्रसामुग्री, पाणी व्यवस्थापन, सुरक्षा, व इतर विविध शेती सुधारणा संबंधित योजना आहेत.

जर आपल्याला शेतभाताच्या कुंपणासाठी (फेन्सिंग) किंवा सौरऊर्जेवर चालणाऱ्या कुंपणासाठी शासकीय अनुदानाची माहिती हवी असेल, तर कृपया आपले जिल्हा किंवा तालुका कृषी कार्यालय, ग्रामपंचायत, किंवा निकटच्या कृषी विज्ञान केंद्रात संपर्क साधा. काही वेळा राज्य किंवा केंद्र सरकारकडून विशेष निधी, समिती, किंवा जिल्हास्तरावर प्रायोगिक योजना जाहीर केल्या जातात, त्या बद्दल स्थानिक कृषी अधिकार्‍यांशी विचारणा करा.

आपल्याला अन्य कोणती शासकीय योजना किंवा अनुदानाची माहिती हवी असेल का? मी महाराष्ट्रातील उपलब्ध कृषी योजनांची अधिकृत माहिती सुद्धा देऊ शकतो.

**स्रोत: शासकीय योजना माहिती** """

In [ ]:
# import import_ipynb
from notebooks import *

In [ ]:
eval_result = await evaluation_agent.run(text)

In [ ]:
print(json.dumps(eval_result.output.model_dump(), indent=2, ensure_ascii=False))

In [ ]:
text_str = text[0] if isinstance(text, list) else text

query = text_str.split("User Query:")[1].splitlines()[0].strip()
language = text_str.split("Language:")[1].splitlines()[0].strip()
response = text_str.split("Final Response:")[1].strip()

df_new = pd.DataFrame.from_dict(
    eval_result.output.model_dump(),
    orient="index"
).reset_index(drop=True)

df_new.insert(0, "query", query)
df_new.insert(1, "response", response)
df_new.insert(2, "language", language)

In [ ]:
output_file = "evaluation_sheet.xlsx"

In [ ]:
df_existing = pd.read_excel(output_file)

In [ ]:
df_final = pd.concat([df_existing, df_new], ignore_index=True)

In [ ]:
df_final.to_excel(output_file, index=False)

Data from models data.json

In [90]:
json_file = "data/models/mistralai_Mistral-Small-3.2-24B-Instruct-2506/data.json"

In [91]:
with open(json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [92]:
len(data)

152

In [93]:
item = data[5]

In [94]:
item

{'question': 'नियंत्रित (Cold/Controlled) गोदाम उपलब्ध आहे का?',
 'category': 'Warehouse',
 'agristack_required': 'Yes',
 'farmer_id': '62891426784',
 'answer': '**MahaVistaar:** "कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो."\n\n**Next Prompt:** "कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो."',
 'agent_turns': [{'parts': [{'tool_name': 'fetch_agristack_data',
     'args': '{}',
     'tool_call_id': 'chatcmpl-tool-b0d138e011454afc',
     'part_kind': 'tool-call'}],
   'usage': {'requests': 1,
    'request_tokens': 9516,
    'response_tokens': 10,
    'total_tokens': 9526,
    'details': {}},
   'model_name': 'mistralai/Mistral-Small-3.2-24B-Instruct-2506',
   'timestamp': '2025-12-30T10:18:35Z',
   'kind': 'response'},
  {'parts': [{'tool_name': 'fetch_agristack_data',
     'content': '> Farmer Information (Agristack)\nNo farmer information found for the requested ID.',
     'tool_call_id': 'chatcmpl-tool-b0d138e011454afc',
  

In [95]:
def response(item):
    lines = []

    lines.append(f"User Query: {item['question']}")
    lines.append("Language: Marathi")
    lines.append("=" * 50 + "\n")

    # Tool calls & returns
    for turn in item.get("agent_turns", []):
        for part in turn.get("parts", []):
            if part["part_kind"] == "tool-call":
                lines.append(f"Assistant called tool: {part['tool_name']}")
                lines.append(f"Arguments: {part.get('args', '{}')}\n")

            elif part["part_kind"] == "tool-return":
                lines.append(f"Tool '{part['tool_name']}' returned:")
                lines.append(part.get("content", "").strip() + "\n")

    lines.append("=" * 50)
    lines.append("=" * 50 + "\n")

    lines.append("Final Response:")
    lines.append(item.get("answer", "").strip())

    return "\n".join(lines)

print(response(item))

User Query: नियंत्रित (Cold/Controlled) गोदाम उपलब्ध आहे का?
Language: Marathi

Assistant called tool: fetch_agristack_data
Arguments: {}

Tool 'fetch_agristack_data' returned:
> Farmer Information (Agristack)
No farmer information found for the requested ID.


Final Response:
**MahaVistaar:** "कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो."

**Next Prompt:** "कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो."


In [96]:
eval_result = await evaluation_agent.run(response(item))

In [97]:
print(json.dumps(eval_result.output.model_dump(), indent=2, ensure_ascii=False))

{
  "source_attribution": {
    "farmer_id": "unknown",
    "question": "नियंत्रित (Cold/Controlled) गोदाम उपलब्ध आहे का?",
    "response": "MahaVistaar: \"कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो.\"\n\nNext Prompt: \"कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो.\"",
    "metric_name": "source_attribution",
    "score": 0,
    "explanation": "No sources or references are cited. The response is only a clarification request and does not attribute information to any authoritative source. Therefore, source attribution is absent."
  },
  "grammatical_accuracy": {
    "farmer_id": "unknown",
    "question": "नियंत्रित (Cold/Controlled) गोदाम उपलब्ध आहे का?",
    "response": "MahaVistaar: \"कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो.\"\n\nNext Prompt: \"कृपया आपली गावाचे नाव सांगा, तेव्हा मी नियंत्रित गोदामांची माहिती देऊ शकतो.\"",
    "metric_name": "grammatical_accuracy",
    "score": 3,
    "exp